# Material Bins Allocation & Combination Optimization (open-source: PuLP + HiGHS)

This notebook is a thin demo of the `binopt` package. All modelling lives in `binopt/`, so the notebook, CLI and tests run the same code.

* **Solver:** [HiGHS](https://highs.dev) (MIT licence) through PuLP. No Gurobi licence is needed.
* **Problem 1, allocation:** one MILP per item. It minimizes pick lines, then fragments created, then bins opened, in lexicographic order.
* **Problem 2, consolidation:** empties fragment bins completely into bins of the same item that have spare room (untouched, partially picked or other fragments). A fragment may be split over at most 2 receiving bins (`max_split`). It maximizes bins freed, then minimizes remaining fragments, then moves, then quantity moved. Chaining is not allowed.

In [1]:
import pandas as pd
from binopt import Settings, load_input, stock_check, select_fulfillable_orders, solve_allocation, solve_consolidation, export_excel, run_pipeline
from binopt.verify import check_allocation, check_consolidation

settings = Settings(
    default_capacity=3000,                # bin capacity when the sheet has no 'bin_capacity' column
    fragment_threshold=1000,              # 0 < left-over < 1000  ->  fragment
    avoid_fragments_in_allocation=True,   # stage 2 of Problem 1
    allow_full_bin_receivers=True,        # fragments may top up partial/untouched bins
    max_split=2,                          # a fragment may be split over at most 2 receiving bins
    solver="highs", time_limit=60, tiebreak_time_limit=15,
)

# 1. Input data

In [2]:
demand, inventory = load_input("input_template.xlsx", settings)
print(f"{demand.production_order.nunique()} orders, {demand.item_code.nunique()} items, {len(inventory)} bins")
demand.head()

16 orders, 6 items, 1791 bins


,production_order,item_code,material_requirements,priority
0,PO1,A57,14219,1.0
1,PO1,A1,12220,1.0
2,PO1,A7,7582,1.0
3,PO2,A57,15868,1.0
4,PO2,A6,12000,1.0


## 1.1 Stock check and order selection
This step replaces the Excel `XLOOKUP` in `stock_check`. If an item is short, a small MILP keeps the orders that maximize priority and then quantity. Orders are kept or dropped whole.

In [3]:
display(stock_check(demand, inventory))
accepted, rejected = select_fulfillable_orders(demand, inventory, settings)
print("rejected orders:", rejected.production_order.tolist())

,item_code,demand,stock,enough_stock
0,A1,62687,64009,True
1,A2,49001,62723,True
2,A57,60390,61824,True
3,A58,60258,61210,True
4,A6,62328,63198,True
5,A7,48347,63432,True


rejected orders: []


# 2. Problem 1: bin allocation

In [4]:
alloc = solve_allocation(accepted, inventory, settings)
print("validation:", check_allocation(accepted, inventory, alloc.allocation) or "OK")
alloc.summary

validation: OK


,item_code,demand,allocated,pick_lines,bins_opened,bins_emptied,fragments_created,bins_available,solver_status
0,A1,62687,62687,27,27,23,3,27,Optimal/TimeLimit/Optimal
1,A2,49001,49001,20,20,16,4,28,Optimal/TimeLimit/Optimal
2,A57,60390,60390,28,28,25,3,28,Optimal/TimeLimit/Optimal
3,A58,60258,60258,28,28,24,4,28,Optimal/TimeLimit/Optimal
4,A6,62328,62328,28,28,22,6,28,Optimal/TimeLimit/Optimal
5,A7,48347,48347,20,20,15,5,28,Optimal/TimeLimit/Optimal


In [5]:
alloc.allocation.head(10)

,production_order,item_code,bin_no,amount_allocated
0,PO11,A1,1,2001
1,PO1,A1,2,2155
2,PO1,A1,3,2428
3,PO11,A1,4,2391
4,PO10,A1,5,2720
5,PO1,A1,6,2162
6,PO10,A1,7,2536
7,PO9,A1,8,2558
8,PO10,A1,9,2348
9,PO9,A1,10,2361


In [6]:
alloc.bin_status.status.value_counts()

status
emptied      125
fragment      25
untouched     16
partial        1
Name: count, dtype: int64

# 3. Problem 2: bin consolidation

In [7]:
cons = solve_consolidation(alloc.bin_status, settings)
print("validation:", check_consolidation(cons.bin_status, cons.moves) or "OK")
cons.summary

validation: OK


,item_code,fragments_before,bins_freed,moves,split_fragments,quantity_moved,fragments_remaining,occupied_bins_before,occupied_bins_after,solver_status
0,A1,3,3,3,0,110,0,4,1,Optimal/Optimal/Optimal/Optimal
1,A2,4,4,4,0,817,0,12,8,Optimal/Optimal/Optimal/Optimal
2,A57,3,2,2,0,437,0,3,1,Optimal/Optimal/Optimal/Optimal
3,A58,4,3,3,0,444,1,4,1,Optimal/Optimal/Optimal/Optimal
4,A6,6,5,5,0,611,1,6,1,Optimal/Optimal/Optimal/Optimal
5,A7,5,5,5,0,1105,0,13,8,Optimal/Optimal/Optimal/Optimal


In [8]:
cons.moves

,item_code,origin_bin,destination_bin,quantity_moved,destination_before,destination_after,destination_capacity
0,A1,17,27,95,1212,1307,3000
1,A1,22,27,2,1307,1309,3000
2,A1,23,27,13,1309,1322,3000
3,A2,319,311,106,1577,1683,3000
4,A2,318,314,100,1567,1667,3000
5,A2,325,323,36,1594,1630,3000
6,A2,312,326,575,1560,2135,3000
7,A57,1459,1472,254,997,1251,3000
8,A57,1479,1472,183,1251,1434,3000
9,A58,1492,1486,35,508,543,3000


# 4. KPIs and export

In [9]:
result = run_pipeline(demand, inventory, settings)
assert not result.errors, result.errors
display(pd.Series(result.kpis(), name="value").to_frame())
export_excel(result, "output_template.xlsx")

,value
orders_served,16.00
orders_rejected,0.00
pick_lines,151.00
bins_opened,151.00
bins_emptied,125.00
fragments_created,25.00
consolidation_moves,22.00
bins_freed_by_consolidation,22.00
fragments_remaining,2.00
pick_stage_not_proven_optimal,0.00
